<!-- FULL: keep, DEMO: keep -->
<div style='display:flex; align-items:center; justify-content:space-between; border-bottom: 3px solid rgb(255,106,0); padding-bottom:1em; margin-bottom:1em'>
<div>
<span style='color:rgb(22,60,105); font-size:1.8em; font-weight:bold;'>Introduction to Deep Learning</span><br>
<span style='color:rgb(0,85,100); font-size:1.3em;'>Session 6 &mdash; 3: Data Pipeline &mdash; Dataset &amp; DataLoader</span><br>
<span style='color:rgb(0,85,100); font-size:1.0em;'>Magda Gregorová &nbsp;·&nbsp; THWS &nbsp;·&nbsp; May 2026</span>
</div>
<img src='../../Common/Pics/thws-logo_vert_en_orange-rgb.png' style='height:80px;'/>
</div>

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import sys

assignment_path = '../../Assignment_1/'
sys.path.append(assignment_path)
from helpers import load_data

X, y = load_data(assignment_path + 'data.csv')
print(f'X: {X.shape}, y: {y.shape}')


<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Why a data pipeline?</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
In notebooks 01 and 02 we trained on the full dataset every step — full-batch gradient descent. Session 5 explained why mini-batching is better:

- cheaper updates — no need to process all data per step
- noise from random batches helps escape sharp minima and saddle points

We could slice tensors manually, but `Dataset` and `DataLoader` handle this cleanly and correctly — including shuffling, batching, and reproducibility.


<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Custom Dataset</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Subclass `torch.utils.data.Dataset` and implement two methods:

- `__len__` — number of examples; called by `len(dataset)`
- `__getitem__` — return the $i$-th example; called by `dataset[i]` and by `DataLoader`

These are **dunder methods** (double underscore) — Python calls them automatically in response to `len()` and indexing. The same pattern was used in Assignment 2 for the `Model` class.


In [ ]:
class HousingDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
dataset = HousingDataset(X, y)
print(f'size: {len(dataset)}')

In [ ]:
# access individual examples by index
x0, y0 = dataset[0]
print(f'x0: {x0}')
print(f'y0: {y0.item():.4f}')

In [ ]:
# slicing also works
X_batch, y_batch = dataset[0:4]
print(f'X_batch shape: {X_batch.shape}')
print(f'y_batch shape: {y_batch.shape}')

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(0,85,100); padding: 0.3em 0.8em; background: rgb(235,245,247); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(0,85,100); font-size:1.05em;'>&#8644; Alternatives &mdash; TensorDataset shortcut</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
When data already fits in memory as tensors, `TensorDataset` wraps them without subclassing — identical behaviour:

```python
dataset = TensorDataset(X, y)
```

Use a custom `Dataset` subclass when data is too large to fit in memory, loaded from disk, or needs on-the-fly preprocessing.


In [ ]:
dataset_td = TensorDataset(X, y)
print(f'size: {len(dataset_td)}')
x0, y0 = dataset_td[0]
print(f'x0: {x0}')

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Train / validation split</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
`random_split` divides a dataset into non-overlapping subsets. Sizes can be given as integers or fractions.

Always split **before** fitting any normalisation statistics — statistics must be computed on training data only (Session 5).


In [ ]:
torch.manual_seed(0)
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2])
print(f'train: {len(train_dataset)}')
print(f'val: {len(val_dataset)}')

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(0,85,100); padding: 0.3em 0.8em; background: rgb(235,245,247); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(0,85,100); font-size:1.05em;'>&#8644; Alternatives &mdash; manual index split</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Full control over which examples go where:

```python
n = len(dataset)
idx = torch.randperm(n)
train_ds = torch.utils.data.Subset(dataset, idx[:int(0.8*n)])
val_ds   = torch.utils.data.Subset(dataset, idx[int(0.8*n):])
```


<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>DataLoader</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
`DataLoader` wraps a `Dataset` and handles:

- splitting into mini-batches of size `batch_size`
- shuffling at the start of each epoch (set `shuffle=True` for training)
- parallel data loading via `num_workers`

Always `shuffle=True` for training — shuffling breaks correlations between consecutive batches and ensures each epoch sees a different ordering.


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

print(f'batches per epoch (train): {len(train_loader)}')
print(f'batches per epoch (val): {len(val_loader)}')

In [ ]:
# inspect one batch
X_batch, y_batch = next(iter(train_loader))
print(f'X_batch shape: {X_batch.shape}')
print(f'y_batch shape: {y_batch.shape}')

<!-- FULL: keep, DEMO: delete -->
Shuffling changes batch contents each epoch:

In [ ]:
# first index in batch differs across epochs
for epoch in range(3):
    X_b, _ = next(iter(train_loader))
    print(f'epoch {epoch}: first example = {X_b[0, :2]}')

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(179,27,27); padding: 0.3em 0.8em; background: rgb(255,240,240); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(179,27,27); font-size:1.05em;'>&#9888; Failure case &mdash; shuffle=True on validation loader</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Setting `shuffle=True` on the validation loader wastes compute and makes results harder to reproduce — validation order does not matter.


<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Training with DataLoader and validation</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Now we bring everything together. The training loop iterates over `train_loader` — one mini-batch per step, one full pass per epoch.

After each epoch we compute the **validation loss** on `val_loader` using `torch.no_grad()`. This is our first look at the train/val gap — the difference between how well the model fits training data vs unseen data.


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        self.layer1 = nn.Linear(in_features, hidden)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden, out_features)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

In [ ]:
torch.manual_seed(0)
model   = MLP(in_features=4, hidden=16, out_features=1)
loss_fn = nn.MSELoss()
lr      = 0.0001

In [ ]:
train_losses = []
val_losses   = []

for epoch in range(100):

    # training
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        with torch.no_grad():
            for p in model.parameters():
                p -= lr * p.grad
        model.zero_grad()
        epoch_loss += loss.item() * len(X_batch)
    train_losses.append(epoch_loss / len(train_loader.dataset))

    # validation
    epoch_val = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            loss = loss_fn(model(X_batch), y_batch)
            epoch_val += loss.item() * len(X_batch)
    val_losses.append(epoch_val / len(val_loader.dataset))

print(f'final train loss: {train_losses[-1]:.4f}')
print(f'final val loss: {val_losses[-1]:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(train_losses, color='#005564', linewidth=2, label='train')
ax.plot(val_losses,   color='#FF6A00', linewidth=2, label='val', linestyle='--')
ax.set_xlabel('epoch')
ax.set_ylabel('MSE loss')
ax.set_title('Train vs validation loss', color='#163C69', fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()


<!-- FULL: keep, DEMO: delete -->
Reading the loss curves:

- **both losses decrease** — training is working
- **val loss > train loss** — expected; model has not seen validation data
- **val loss stops improving while train loss continues** — overfitting
- **val loss spikes** — learning rate too large or batch too small

The manual update loop works but is verbose. Notebook 04 introduces `torch.optim` which handles the update step cleanly and adds momentum, Adam, and learning rate schedules with one line.
